## Training and measuring performance for L1 and L2 regularizers

In [ ]:
from util import *
import numpy as np
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# Data generation
np.random.seed(2025)
n_points = 1000

noise = 0.01 * np.random.randn(n_points,1)
x = np.linspace(-1, 1, n_points).reshape(-1,1)
y = Runge(x) + noise

degree = 14
X = PolynomialFeatures(degree).fit_transform(x)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25)

# Data scaling
scaler = StandardScaler()
scaler.fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)
y_offset = np.mean(y_test)

# Do a bit of a hack to work with the exact same data as the regression does :)
x_train_s = X_train_s[:,1].reshape(-1,1)
x_test_s = X_test_s[:,1].reshape(-1,1)
y_train = y_train.reshape(-1,1)
y_test = y_test.reshape(-1,1)


lambdas = [10**(-6), 10**(-4), 10**(-2), 1]
learning_rates = [0.0001, 0.001, 0.01, 0.05]

l1_mses = np.zeros((len(lambdas), len(learning_rates)))
l1_r2s = np.zeros((len(lambdas), len(learning_rates)))

l2_mses = np.zeros((len(lambdas), len(learning_rates)))
l2_r2s = np.zeros((len(lambdas), len(learning_rates)))

for i, lmbd in enumerate(lambdas):
	for j, lr in enumerate(learning_rates):
		print(f"Training Lambda: {lmbd}, Learning rate: {lr}")
		
		# L1
		nn = NeuralNetwork(1, [100, 100, 100, 100, 1], [ReLU, ReLU, ReLU, ReLU, linear], [ReLU_der, ReLU_der, ReLU_der, ReLU_der, linear_der], MSE_L1, MSE_L1_der, lam = lmbd, regularizer="L1")
		nn.ADAM_stochastic(x_train_s, y_train - y_offset, learning_rate=lr, rho1 = 0.9, rho2 = 0.999, epochs = 500, minibatch_size=5)
		nn_pred = nn.predict_batch(x_test_s) + y_offset 
		mse = mean_squared_error(y_test, nn_pred)
		r2 = r2_score(y_test, nn_pred)
		l1_mses[i, j] = mse
		l1_r2s[i, j] = r2
		print("L1 MSE:", mse, "R2:", r2,"\n")
		
		# L2 regularization
		nn = NeuralNetwork(1, [100, 100, 100, 100, 1], [ReLU, ReLU, ReLU, ReLU, linear], [ReLU_der, ReLU_der, ReLU_der, ReLU_der, linear_der], MSE_L2, MSE_L2_der, lam = lmbd, regularizer="L2")
		nn.ADAM_stochastic(x_train_s, y_train - y_offset, learning_rate=lr, rho1 = 0.9, rho2 = 0.999, epochs = 500, minibatch_size=5)
		nn_pred = nn.predict_batch(x_test_s) + y_offset
		mse = mean_squared_error(y_test, nn_pred)
		r2 = r2_score(y_test, nn_pred)
		l2_mses[i, j] = mse
		l2_r2s[i, j] = r2
		print("L2 MSE:", mse, "R2:", r2, "\n")

## Plotting

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(dpi=300)
# Remove first column for L1 as it gave inf MSEs
sns.heatmap(l1_mses[:,1:], xticklabels=learning_rates[1:], yticklabels=lambdas, annot=True, fmt=".4f")
plt.xlabel("Learning rate")
plt.ylabel(r"$\lambda$")
plt.title("L1 Regularization: MSE for different hyperparameters")
plt.show()
plt.figure(dpi=300)
sns.heatmap(l2_mses[:,1:], xticklabels=learning_rates[1:], yticklabels=lambdas, annot=True, fmt=".4f")
plt.xlabel("Learning rate")
plt.ylabel(r"$\lambda$")
plt.title("L2 Regularization: MSE for different hyperparameters")

## Ridge and LASSO regression

In [ ]:
import numpy as np
from util import *
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_squared_error
def Ridge(X:np.ndarray, y:np.ndarray, l)->np.ndarray:
    return np.linalg.pinv(X.T @ X + l * np.identity(len(X[0]))) @ X.T @ y

np.random.seed(2025)
n_points = 1000

noise = 0.01 * np.random.randn(n_points,1)
x = np.linspace(-1, 1, n_points).reshape(-1,1)
y = Runge(x) + noise

degree = 11
X = PolynomialFeatures(degree).fit_transform(x)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25)

scaler = StandardScaler()
scaler.fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)
y_offset = np.mean(y_test)

gd_ridge = gradient_descent(X_train_s, y_train, Ridge_Gradient, learning_rate=0.001, hyperparameter=1e-6)
theta_ridge = gd_ridge.ADAM_stochastic(0.9, 0.999, 500, 5)
y_pred_ridge = X_test_s @ theta_ridge + y_offset
mse_ridge = mean_squared_error(y_test, y_pred_ridge)
print(f"Ridge Regression MSE: {mse_ridge}")

gd_lasso = gradient_descent(X_train_s, y_train, LASSO_Gradient, learning_rate=0.001, hyperparameter=1e-4)
theta_lasso = gd_lasso.ADAM_stochastic(0.9, 0.999, 500, 5)
y_pred_lasso = X_test_s @ theta_lasso + y_offset
mse_lasso = mean_squared_error(y_test, y_pred_lasso)
print(f"Lasso Regression MSE: {mse_lasso}")